# 07 - Full Pipeline Summary

Ties notebooks 01-06 together: loads every saved artifact, runs the
complete recommendation flow for a few example playlists, and summarizes
what this project does and doesn't demonstrate yet.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import pickle
from src.models.fallback_handler import SpotifyColdStartRouter

tracks = pd.read_csv('../data/processed/tracks_clean.csv')
tracks['track_uri'] = 'spotify:track:' + tracks['track_id'].astype(str)
interactions_df = pd.read_csv('../data/processed/playlist_track_interactions.csv')

with open('../data/processed/id_mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)
track_to_idx = mappings['track_to_idx']

with open('../data/processed/lightfm_model.pkl', 'rb') as f:
    hybrid_model = pickle.load(f)

content = np.load('../data/processed/content_embeddings.npz', allow_pickle=True)
emb_uris = list(content['track_uris'])
emb_matrix = content['embeddings']
uri_to_emb_idx = {u: i for i, u in enumerate(emb_uris)}

candidate_df = tracks[['track_uri', 'track_name', 'artists', 'album_name', 'popularity']].rename(columns={'artists': 'artist_name'})
candidate_df = candidate_df[candidate_df['track_uri'].isin(track_to_idx) & candidate_df['track_uri'].isin(uri_to_emb_idx)].reset_index(drop=True)
candidate_df['track_idx'] = candidate_df['track_uri'].map(track_to_idx)

interaction_counts = interactions_df.groupby('track_uri').size().rename('interaction_count')
candidate_df['interaction_count'] = candidate_df['track_uri'].map(interaction_counts).fillna(0).astype(int)

cand_emb_idx = candidate_df['track_uri'].map(uri_to_emb_idx).values
cand_embeddings = emb_matrix[cand_emb_idx]
sim_matrix = cand_embeddings @ cand_embeddings.T
local_idx_map = {uri: i for i, uri in enumerate(candidate_df['track_uri'])}
candidate_df['local_idx'] = candidate_df['track_uri'].map(local_idx_map)

router = SpotifyColdStartRouter(warm_playlist_threshold=5, warm_track_threshold=10)
print("Loaded", len(candidate_df), "candidate tracks and the trained hybrid model.")

Loaded 16804 candidate tracks and the trained hybrid model.


## Recommend for three different real (genre-derived) playlists

In [2]:
def recommend_for_playlist(pid, top_n=10):
    playlist_tracks = interactions_df[interactions_df['pid'] == pid]['track_uri'].tolist()
    playlist_name = interactions_df[interactions_df['pid'] == pid]['playlist_name'].iloc[0]
    track_indices = [local_idx_map[u] for u in playlist_tracks if u in local_idx_map]

    recs = router.route_recommendations(
        playlist_idx=pid,
        playlist_track_count=len(playlist_tracks),
        candidate_tracks=candidate_df.assign(track_idx=candidate_df['local_idx']),
        hybrid_model=hybrid_model,
        content_similarity_matrix=sim_matrix,
        playlist_track_indices=track_indices,
    )
    result = candidate_df.set_index('track_uri').loc[recs[:top_n], ['track_name', 'artist_name', 'popularity']]
    print(f"Playlist: {playlist_name} ({len(playlist_tracks)} tracks)")
    return result

sample_pids = interactions_df['pid'].drop_duplicates().sample(3, random_state=7).tolist()
for pid in sample_pids:
    display(recommend_for_playlist(pid))
    print()

Playlist: Latino Mix 407 (334 tracks)


,track_name,artist_name,popularity
track_uri,,,
spotify:track:0kKuVSnv97w8dZ0yjde1KU,No compasso do criador - Ao vivo,Katinguelê,39
spotify:track:1bngrz3hq1AIdAD5wmzHTr,Minha Fama de Mau - Ao Vivo,Erasmo Carlos,36
spotify:track:43GJmTLEE0itzVPFmCKmmS,Sentado à Beira do Caminho - Ao Vivo,Erasmo Carlos,39
spotify:track:3XZi8w4xJLZp5wOfcabUje,Bad Boys - Fullerton Mix,DJ Sneak;Bad Boy Bill,18
spotify:track:3LaQzvfIetBWdtIV0JoBwF,COMO UN BEBÉ,J Balvin;Bad Bunny;Mr Eazi,4
spotify:track:2uJ8FAeLomeAqLejkIQQu4,Múmias,Biquini Cavadão,39
spotify:track:2dTeSx3Ve6tIum4T7haUWO,Ahora Me Llama,KAROL G;Bad Bunny,4
spotify:track:02zFCfzkqFwNaZ4f0idBbi,Me Apaixonei Pela Pessoa Errada - Ao Vivo,Exaltasamba,39
spotify:track:1Gp3235UGtWR50HwDFDjyu,Heart Of Pop (Percayalah),Closehead,38


Playlist: Opera Mix 458 (346 tracks)


,track_name,artist_name,popularity
track_uri,,,
spotify:track:1PH1U3ZfSoKP0hB5OuVR6H,New Found Love,Popcaan,30
spotify:track:0kKuVSnv97w8dZ0yjde1KU,No compasso do criador - Ao vivo,Katinguelê,39
spotify:track:1bngrz3hq1AIdAD5wmzHTr,Minha Fama de Mau - Ao Vivo,Erasmo Carlos,36
spotify:track:1m37Hf4mCV6XoGA1AuSfR9,Dinleyiverin Gayri,Moğollar,39
spotify:track:4tGNtRblzjwLkHllmH8rIS,Çaya Kaç Şeker,Moğollar,39
spotify:track:339jxAN4N3d1kBp6WGxmlL,夜間飛行少年,Qujila Yoluno Machi,38
spotify:track:4aoN6rHl9SkAZJw8BFh4yf,Nefesim Nefesine,Zülfü Livaneli,38
spotify:track:3v58FgWXLqxLIAkO8tG43N,Kan Çiçekleri,Zülfü Livaneli,37
spotify:track:0l3p85cVj6FVQ2Nh7l94sr,Benimle Evlensen,Evrencan Gündüz,37


Playlist: Metal Mix 429 (321 tracks)


,track_name,artist_name,popularity
track_uri,,,
spotify:track:1ghbMdugKTOluXxDiIc1Pp,Enlightment,Totalselfhatred,39
spotify:track:5R4YNUrn32685RHjuDg5NK,Leblos,Eisregen,29
spotify:track:7l04WFDhxrgYoiUgiaTL1r,Ins Niemandsland,Kanonenfieber,28
spotify:track:1SNBvk4kYL0WKZfCFu1g4p,Я свободен!,Кипелов,39
spotify:track:3mrCVBnj3LOOKZLlRoj2MT,Kadonneet kolme sanaa,Stam1na,37
spotify:track:1FQG72IMugWnzCQcCblz6j,Partiendo Cabezas,Asspera,36
spotify:track:2BkoAPwO85Zb2IOJF0LMRv,Río Extraño,OConnor,35
spotify:track:4wvqASb0pNFqVHw6rQad2z,Kannoin sinut läpi hiljaisen huoneen,Stam1na,34
spotify:track:5qT9INHPbJzRp3PgfM2I84,Elämänlanka,Stam1na,34


## Project summary

**What's real:**

- The 16.8k-track catalog (`data/raw/spotify_tracks_dataset.csv`) is a real,
  publicly available Spotify track dataset with genuine audio features,
  genres, and popularity scores.
- The `InteractionMatrixBuilder`, `SpotifyHybridCollaborative` (LightFM), and
  `SpotifyColdStartRouter` classes are the actual project code, exercised
  end-to-end in these notebooks - not reimplemented or mocked.
- The LightFM training and evaluation numbers (notebook 05) are genuine
  results on genuine (if synthetic-playlist) sparse interaction data,
  including the honest train/test generalization gap and what's driving it.
- A real bug in `fallback_handler.py`'s content-based cold-start scoring was
  found and fixed as part of this project (notebook 06).

**What's simulated, and why:**

- **Playlists** are constructed from real tracks grouped by genre (notebook
  02), not real Spotify user playlists. Real playlist logs come from the
  Million Playlist Dataset, which requires registering for restricted
  access through Spotify/AIcrowd and is too large to fetch automatically.
  Swap in real MPD slices and every notebook downstream keeps working
  unchanged, since everything only depends on `pid` / `track_uri` columns.
- **Content embeddings** use TF-IDF + SVD (notebook 03) instead of the
  `sentence-transformers` SBERT model that `src/features/content_embeddings.py`
  uses in production, purely because downloading SBERT's model weights
  needs outbound internet access to Hugging Face that this environment
  doesn't have. The production code path is unchanged and works normally
  with regular internet access.

**Honest limitations:**

- Test-set collaborative-filtering performance (AUC ~0.54, precision@10
  ~0.004) is modest - a direct consequence of genre-chunked synthetic
  playlists having less real co-listening signal than actual user behavior.
  This is the single biggest lever for improving the project: real
  interaction data.
- Content embeddings here are TF-IDF-based, which only captures lexical
  overlap in track/artist/album names - SBERT would pick up on semantic
  similarity even between differently-worded but related tracks.